In [1]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import os

print("gseapy version:", gp.__version__)

gseapy version: 1.1.11


In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

sub_out = "hapatovirus_260113_MSigDB_Hallmark_2020_stat/"  # 260113
out_folder_path = f"{basedir}/output/251225GSEA/"
out_path = out_folder_path + sub_out

input_folder_path = f"{basedir}/data/251215_rna_seq/To_kurihara_hepatovirus/output/"

# make directories
if not(os.path.exists(out_folder_path)):
    os.mkdir(out_folder_path)
if not(os.path.exists(out_path)):
    os.mkdir(out_path)

In [3]:
def plot_prerank(df, outdir_path, metric, gene_col, gene_sets):
    """
    prerank tool plot.

    metric: ranking metric
    gene_col: gene name col for enrichr
    """
    # cols: gene_name_hs, gene_name_gallus, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj    
    df2 = df.copy()
    
    # drop NaN
    df2 = df2.dropna(subset=[gene_col, metric])
    df2[gene_col] = df2[gene_col].astype(str)
    # df2[gene_col] = df2[gene_col].astype(str).str.upper()
    
    # for duplicate genes, keep the one with the larger absolute value 
    # (for ties: 1. pvalue, 2. gene name)    df2["_abs"] = df2[metric].astype(float).abs()
    df2["_p"] = df2["pvalue"].astype(float).fillna(1.0)

    df2 = (df2.sort_values([gene_col, "_abs", "_p", gene_col],
                           ascending=[True, False, True, True],
                           kind="mergesort").drop_duplicates(subset=[gene_col], keep="first"))

    # ranking (for ties: 1. pvalue, 2. gene name)
    df2 = df2.sort_values([metric, "_p", gene_col],
                          ascending=[False, True, True],
                          kind="mergesort")
    rnk = df2[[gene_col, metric]]
    print("\nrnk\n", rnk.head(3))

    # prerank
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        outdir=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}",
        seed=0,
        threads=4,
        verbose=True
    )

    # plot top 5 pathways
    terms = pre_res.res2d["Term"]
    pre_res.plot(
        terms=terms[:5], 
        legend_kws={'loc': (1.15, 0)},
        ofname=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}_top5_enriched.png"
    )

In [4]:
# check Enricher library
names = gp.get_library_name()
# print("all library:", names)
print("KEGG library:", [k for k in names if k.startswith("KEGG_")])
print("MSigDB library:", [m for m in names if m.startswith("MSigDB_")])
print("Reactome library:", [r for r in names if r.startswith("Reactome")])
print("GO library:", [g for g in names if g.startswith("GO_Biological")])

KEGG library: ['KEGG_2013', 'KEGG_2015', 'KEGG_2016', 'KEGG_2019_Human', 'KEGG_2019_Mouse', 'KEGG_2021_Human']
MSigDB library: ['MSigDB_Computational', 'MSigDB_Hallmark_2020', 'MSigDB_Oncogenic_Signatures']
Reactome library: ['Reactome_2022', 'Reactome_Pathways_2024']
GO library: ['GO_Biological_Process_2021', 'GO_Biological_Process_2023', 'GO_Biological_Process_2025']


In [5]:
configs = {
    "metric": "stat",
    "gene_col": "gene_name_hs",
    "gene_sets": "MSigDB_Hallmark_2020"
}

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if not(fname.startswith("result")):
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)

        # plot prerank
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)
        plot_prerank(project, out_path + proname, configs["metric"], configs["gene_col"], configs["gene_sets"])

2026-01-13 09:15:45,104 [WARNING] Duplicated values found in preranked stats: 1.38% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-13 09:15:45,104 [INFO] Parsing data files for GSEA.............................
2026-01-13 09:15:45,113 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-01-13 09:15:45,118 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-13 09:15:45,119 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-13 09:15:45,119 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA774885

rnk
       gene_name_hs      stat
19233          PI3  7.624489
14275       STOML1  6.930276
9447         SPON2  5.743817


2026-01-13 09:15:49,237 [INFO] Congratulations. GSEApy runs successfully................

2026-01-13 09:15:49,501 [WARNING] Duplicated values found in preranked stats: 1.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-13 09:15:49,502 [INFO] Parsing data files for GSEA.............................
2026-01-13 09:15:49,510 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-01-13 09:15:49,515 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-13 09:15:49,516 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-13 09:15:49,516 [INFO] Start to run GSEA...Might take a while..................



processing... PRJEB63475

rnk
       gene_name_hs       stat
16764         XPO6  12.645954
3728        PRKAG2  12.602400
4515          BRD9  11.994078


2026-01-13 09:15:54,850 [INFO] Congratulations. GSEApy runs successfully................

